# TN_test — cấu hình 192 kênh từ thí nghiệm cũ có đáng dùng không

## Đây là phép thăm dò, không phải một dòng của TN1

Chạy **một seed** để quyết định có đáng đầu tư ba seed hay không. Nếu đáng thì
mới nâng lên đủ ba seed và đưa vào bảng chính.

## Vì sao thăm dò

TN1 có một chỗ hổng thật, và hội đồng hỏi được:

> Mọi cấu hình TN1 chạy bằng siêu tham số của `optimal_params.json`, vốn được
> MobiVital dò riêng cho LSTM-352. Bảy kiến trúc tích chập đều chạy ở thiết lập
> tối ưu cho một kiến trúc khác. Vậy TCN thua thật, hay chỉ là chưa được chỉnh?

Thí nghiệm cũ tháng 8 (`old_expreiment_outdated_donotuse/`) đã dò Optuna riêng
cho TCN, và ra một cấu hình to hơn hẳn: **192 kênh, 4 khối**, khoảng 310 nghìn
tham số. Trên GHIJ nó đạt **0,8086 ± 0,0127**, ngang LSTM-352 (0,8103) với năm
lần ít tham số hơn, và hơn DS-TCN-64 của TN1 (0,7958).

Điểm GHIJ đó **sạch** — GHIJ chưa bao giờ dùng để chọn cấu hình ở cả hai bên.
Còn điểm trên KL của thí nghiệm cũ thì **không dùng được**, vì Optuna đã tối ưu
trên chính KL.

## Cấu hình cũ dựng lại được tới đâu

    thí nghiệm cũ, RF121      kernel 5, 4 khối, 192 kênh, tầm nhìn 121
                              KHÔNG có lớp chuẩn hoá nào
                              310.873 tham số

    dựng lại bằng code này    ds_tcn --channels 192 --kernel_size 5
                              --n_blocks 4 --dropout 0.2
                              CÓ BatchNorm
                              313.945 tham số

Chênh đúng **3.072 = 8 lớp BatchNorm × 384**. Bỏ hết BatchNorm khỏi bản dựng
lại thì ra đúng 310.873. Đây là chỗ lệch duy nhất về kiến trúc, khoảng 1% tham
số, và có lý do giữ: `ds_tcn` của TN1 dùng BatchNorm theo Howard et al. 2017
mục 3.1.

Hai thứ của cấu hình cũ **không** dựng lại được, vì `run_cv.py` chưa có cờ:

    learning rate    cũ 2,13e-4    ở đây 1e-4 theo giao thức TN1
    weight decay     cũ 3e-7       ở đây 0
    loss             cũ hỗn hợp    ở đây MSE thuần

Việc bỏ ba thứ đó là **có chủ ý**: giữ nguyên giao thức TN1 thì kết quả so thẳng
được với tám cấu hình kia. Chạy nguyên si cấu hình cũ sẽ đổi sáu thứ cùng lúc,
thắng thua đều không quy được cho cái gì.

## Câu hỏi cụ thể

**DS-TCN thua có phải vì nó quá bé không?**

    DS-TCN-64      56.281 tham số     cv 0,7421 ± 0,0007
    DS-TCN-192    313.945 tham số     ?

Đổi ba thứ: số kênh 64 → 192, số khối 6 → 4, dropout 0 → 0,2. Không phải một
biến, nên nếu thắng thì chưa quy được cho riêng số kênh. Nhưng nếu **không**
thắng thì kết luận rõ ràng: sức chứa không phải thứ đang thiếu.

## Một chi tiết đáng ghi về tầm nhìn

    TN1 tcn/ds_tcn     kernel 3, 6 khối    tầm nhìn 253    phủ trọn 200 mẫu
    cũ RF121           kernel 5, 4 khối    tầm nhìn 121    thấy 121/200 mẫu
    cũ RF61            kernel 3, 4 khối    tầm nhìn  61    thấy  61/200 mẫu

Cả hai cấu hình cũ đều **không nhìn hết cửa sổ đầu vào**. Thí nghiệm cũ cũng
thấy điều đó: RF121 hơn RF61 trên GHIJ (0,8086 so với 0,7950). Có thể tầm nhìn
dài hơn nữa còn tốt hơn, nhưng thí nghiệm cũ dừng ở 121.

## Fold val_KL bị nhiễm cho riêng cấu hình này

Cấu hình 192 kênh vốn được Optuna chọn bằng cách tối ưu trên KL. Vì vậy điểm
của nó ở fold `val_KL` có lợi thế không công bằng.

**Đọc ba fold `val_AB`, `val_CE`, `val_DF` tách riêng.** Chúng sạch.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt và đối chiếu số tham số

Xác nhận hai điều trước khi train: model dựng đúng, và số tham số khớp con số
đã tính (313.945, tức 310.873 của thí nghiệm cũ cộng 3.072 của BatchNorm).

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2

## 3. Chạy 4 fold, MỘT seed

Giao thức giữ nguyên TN1: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9,
bốn fold cũ.

Tên cấu hình là `ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0`, nằm trong thư mục
`runs/tn_test/` riêng — không lẫn vào bảng TN1.

8 tầng tích chập ở 192 kênh, nặng hơn DS-TCN-64 khá nhiều. Ước lượng
**khoảng 1,5 giờ** cho bốn fold.

In [ ]:
!python scripts/run_cv.py --experiment tn_test --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn_test --out tn_test_ds_tcn_c192

## 5. Đọc kết quả

`compare_cv` chỉ in được cấu hình trong `runs/tn_test/`, tức đúng một dòng. Để
so với TN1 thì lấy con số `cv_score` in ra ở đây rồi đặt cạnh bảng TN1.

Mốc để so:

| | tham số | cv_mean |
|---|---|---|
| LSTM-352 | 1.502.713 | 0,7570 |
| LSTM-67 | 56.908 | 0,7532 |
| DS-TCN-64 | 56.281 | 0,7421 |

**Nhắc lại: bỏ fold `val_KL` ra khi đọc.** Ba fold còn lại mới sạch cho cấu
hình này.

Quyết định sau khi xem:

    hơn DS-TCN-64 rõ trên ba fold sạch    -> đáng chạy nốt 2 seed, đưa vào bảng
    ngang hoặc kém                        -> sức chứa không phải thứ đang thiếu,
                                             ghi kết quả âm này vào luận văn và
                                             chuyển sang TN3

In [ ]:
!python scripts/compare_cv.py --experiment tn_test

## 6. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()